# Runtime, peak memory and estimates: torchpalmm vs. mgcv `bam` and `gamm` on the SPT model

The model of `fit_logit_noround_BOP_SP.ipynb` (visits 2 onward; outcome $\operatorname{logit}((\text{bleeding \%} + 1)/102)$ with bleeding rounded to 0.1; for each of the 6 baseline class $\times$ sex groups one curve of follow-up time (0.1-year grid) and one curve of bleeding at the first visit (1% grid); fixed effects age, calendar year, teeth, log time since the preceding visit, class, sex, ever-smoker and class $\times$ sex, standardized as in the notebook; a random intercept with its own variance in each class $\times$ sex $\times$ smoker stratum (12 strata) plus random slopes on age at visit, teeth at the visit and log time since the preceding visit, with diagonal $D$; iid residuals) is fitted by torchpalmm (REML, L-BFGS) and by mgcv `bam` (fREML) and `gamm` (nlme REML).

Each fit runs in its own process under `/usr/bin/time -l` (macOS: peak resident set size), and a "baseline" process (start-up, imports and data only) is subtracted. **Every process has a 30-minute wall-clock cap** and is killed at the cap; its time is reported as the cap, and memory and estimates are unavailable. The process reads the prepared sample and builds its own model matrices, so matrix construction is part of the cost.

Differences between the methods: torchpalmm puts a knot at every distinct smooth-input value, and with `K = "match"` mgcv (`bam` and `gamm`) gets the identical basis: per group a `cr` spline with its knots at that group's distinct values (centred, so linear part + penalized part = distinct values - 1 coefficients per curve, as torchpalmm), and the same $\int f''^2$ penalty. The class / sex coefficients depend on how each curve's constant is pinned, so mgcv's are converted to torchpalmm's convention (estimates and SEs) before comparison; All three fit a diagonal $D$: a patient belongs to one stratum, so covariances between the stratum intercepts are not identified. torchpalmm uses 12 stratum-indicator columns in $Z$ plus the three slope columns, `bam` uses `s(pat, by = grp_re, bs = 're')` plus a random-effect smooth per slope, and `gamm` uses `nlme::pdDiag` on the same columns. The smooth curves stay specific to the 6 class $\times$ sex groups. Fixed-effect estimates are compared with torchpalmm's; smooth curves are not.

In [4]:
## settings, analysis sample (same rows, outcome and covariates as fit_logit_noround_BOP_SP.ipynb) and helpers
import subprocess, re, sys, os, signal, time
import numpy as np, pandas as pd
os.makedirs("benchmark/out", exist_ok=True)
data_csv = os.path.abspath("benchmark/spt_analysis_sample_more_re.csv")
CAP = 30 * 60                                                        # wall-clock cap per process (s)
K = "match"                                                          # mgcv basis per curve: "match" = torchpalmm's basis (knots at every distinct value of the group, k = distinct values), or an integer such as 20

d = pd.read_csv("spt_data/curated_spt_bleeding.csv")
d["bop_first"] = d["pat_id"].map(d.loc[d["spt_id"] == 1].set_index("pat_id")["bleeding_pct"])   # BOP (%) at the first visit, constant within patient
d["bleeding_pct"] = (d["bleeding_pct"] / 0.1).round() * 0.1          # outcome rounded to 0.1 (bop_first stays unrounded)
d = d[d["spt_id"] >= 2].copy()                                       # first visit excluded; age 90 kept, no interval filter
d["ever"] = np.where(d["smoking_status"].isna(), np.nan, (d["smoking_status"] != "non-smoker").astype(float))
d["logdays"] = np.log(d["days_since_prev_visit"])
need = ["bleeding_pct", "cum_followup_years", "bop_first", "age_at_spt", "calendar_year", "n_teeth_spt", "logdays", "baseline_diagnosis", "sex", "ever"]
d = d.dropna(subset=need).sort_values(["pat_id", "spt_id"]).reset_index(drop=True)
p = (d["bleeding_pct"] + 1) / 102
s = pd.DataFrame({"pat_id": d["pat_id"], "spt_id": d["spt_id"], "y": np.log(p / (1 - p)),
                  "fu": ((d["cum_followup_years"] / 0.1).round() * 0.1).round(10),                    # follow-up time on the 0.1-year grid
                  "bop_first": (d["bop_first"] / 1.0).round() * 1.0})                                # BOP at first visit on the 1% grid
for k, v in {"age_z": d["age_at_spt"], "year_z": d["calendar_year"], "teeth_z": d["n_teeth_spt"], "logdays_z": d["logdays"]}.items():
    s[k] = (v - v.mean()) / v.std()
cls = d["baseline_diagnosis"].astype(int).astype(str); fem = (d["sex"] == "female").astype(float)
s["cls2"], s["cls3"], s["female"], s["ever"] = (cls == "2").astype(float), (cls == "3").astype(float), fem, d["ever"].astype(float)
s["cls2_female"], s["cls3_female"] = s["cls2"] * fem, s["cls3"] * fem
s["grp"] = "class " + cls + ", " + d["sex"]
s["grp_re"] = s["grp"] + ", " + np.where(s["ever"] == 1, "ever", "never")       # random-intercept stratum: class x sex x smoker
assert s.notna().all().all()
s.to_csv(data_csv, index=False)
print(f"{len(s)} visits, {s['pat_id'].nunique()} patients; distinct values per group: follow-up", s.groupby("grp")["fu"].nunique().to_dict(), "| BOP at first visit", s.groupby("grp")["bop_first"].nunique().to_dict())

ENV1 = {**os.environ, **{v: "1" for v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS", "RAYON_NUM_THREADS")}}   # one thread per process (BLAS, OpenMP, torch) so runtime and memory are comparable

def run(cmd, out_csv=None):
    """Run cmd under /usr/bin/time -l with a CAP-second wall-clock cap; return status, wall seconds, peak memory (GB) and the RESULT line."""
    if out_csv and os.path.exists(out_csv):
        os.remove(out_csv)                                           # never compare against a stale file
    t0 = time.perf_counter()
    proc = subprocess.Popen(["/usr/bin/time", "-l"] + cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, start_new_session=True, env=ENV1)
    try:
        out, err = proc.communicate(timeout=CAP)
    except subprocess.TimeoutExpired:
        os.killpg(proc.pid, signal.SIGKILL); proc.communicate()
        return {"status": "timeout", "wall (s)": float(CAP), "peak memory (GB)": np.nan, "result": ""}
    peak = re.search(r"(\d+)\s+maximum resident set size", err)
    wall = re.search(r"([\d.]+)\s+real", err)
    if proc.returncode or not peak:
        return {"status": "error", "wall (s)": time.perf_counter() - t0, "peak memory (GB)": np.nan, "result": re.split(r"\n\s*[\d.]+\s+real", err)[0][-1500:]}   # the R / Python error, without the time -l statistics
    result = re.search(r"RESULT .*", out)
    return {"status": "ok", "wall (s)": float(wall.group(1)), "peak memory (GB)": int(peak.group(1)) / 1e9, "result": result.group(0) if result else ""}

10860 visits, 870 patients; distinct values per group: follow-up {'class 1, female': 180, 'class 1, male': 157, 'class 2, female': 225, 'class 2, male': 241, 'class 3, female': 231, 'class 3, male': 220} | BOP at first visit {'class 1, female': 35, 'class 1, male': 30, 'class 2, female': 62, 'class 2, male': 52, 'class 3, female': 46, 'class 3, male': 48}


In [5]:
## run: baselines and the three fits (each with its inference summary: SEs and EDFs), each in its own process (at most CAP seconds each)
py = [sys.executable, "benchmark/benchmark_spt_torchpalmm_more_re.py"]
R = ["Rscript", "benchmark/benchmark_spt_mgcv_more_re.R"]
out = {"torchpalmm": "benchmark/out/spt_tp_more_re.csv", "bam": "benchmark/out/spt_bam_more_re.csv", "gamm": "benchmark/out/spt_gamm_more_re.csv"}
base_tp = run(py + ["baseline", data_csv, out["torchpalmm"]])
base_r = run(R + ["baseline", data_csv, out["bam"], str(K)])
fits = {"torchpalmm": run(py + ["summary", data_csv, out["torchpalmm"]], out["torchpalmm"]),
        "bam": run(R + ["bam", data_csv, out["bam"], str(K)], out["bam"]),
        "gamm": run(R + ["gamm", data_csv, out["gamm"], str(K)], out["gamm"])}
base = {"torchpalmm": base_tp["peak memory (GB)"], "bam": base_r["peak memory (GB)"], "gamm": base_r["peak memory (GB)"]}
fit_secs = lambda r: float(re.search(r"fit_seconds=([\d.]+)", r["result"]).group(1)) if "fit_seconds" in r["result"] else np.nan
tab = pd.DataFrame([{"method": m, "status": r["status"], "wall (s)": r["wall (s)"], "fit (s)": fit_secs(r), "peak memory (GB)": r["peak memory (GB)"],
                     "peak minus baseline (GB)": r["peak memory (GB)"] - base[m]} for m, r in fits.items()]).set_index("method")
print(f"cap {CAP} s per process (a timeout is reported at the cap, memory unavailable)")
print(tab.round(2).to_string())
for m, r in fits.items():
    print(m, "|", r["result"])

cap 1800 s per process (a timeout is reported at the cap, memory unavailable)
             status  wall (s)  fit (s)  peak memory (GB)  peak minus baseline (GB)
method                                                                            
torchpalmm       ok    504.71    172.2              2.99                      2.72
bam         timeout   1800.00      NaN               NaN                       NaN
gamm        timeout   1800.00      NaN               NaN                       NaN
torchpalmm | RESULT mode=summary D=diagonal rows=10860 subjects=870 total_basis=1503 smooth_coefs=1515 woodbury=True epochs=18 lbfgs_steps=110 func_evals=208 converged=1 fit_seconds=172.2 summary_seconds=331.4
bam | 
gamm | 


In [6]:
## refresh bam only: refit bam (e.g. after changing its settings) and keep the torchpalmm and gamm results from the run cell
fits["bam"] = run(R + ["bam", data_csv, out["bam"], str(K)], out["bam"])
print("bam |", fits["bam"]["status"], f"| wall {fits['bam']['wall (s)']:.1f} s | peak memory {fits['bam']['peak memory (GB)']:.2f} GB")
print(fits["bam"]["result"])


KeyboardInterrupt: 

In [ ]:
## refresh gamm only: refit gamm (e.g. after changing its settings) and keep the torchpalmm and bam results from the run cell
fits["gamm"] = run(R + ["gamm", data_csv, out["gamm"], str(K)], out["gamm"])
print("gamm |", fits["gamm"]["status"], f"| wall {fits['gamm']['wall (s)']:.1f} s | peak memory {fits['gamm']['peak memory (GB)']:.2f} GB")
print(fits["gamm"]["result"])


gamm | ok | wall 709.8 s | peak memory 6.27 GB
RESULT mode=gamm rows=10860 subjects=870 total_basis=1526 smooth_basis=1515 converged=1 fit_seconds=708.7 summary_seconds=0.1


In [7]:
## results: torchpalmm vs mgcv::gamm and vs mgcv::bam, group-specific random intercepts, diagonal D (Diff = torchpalmm - other method)
from IPython.display import HTML, display
TH = "border:1px solid #000;padding:5px 12px;font:14px Calibri,Arial,sans-serif;"
def html_table(df, section_rows=()):
    h = "<table style='border-collapse:collapse'><tr>" + "".join(f"<th style='{TH}background:#d9d9d9;text-align:center'>{c}</th>" for c in df.columns) + "</tr>"
    for i, row in df.iterrows():
        sec = i in section_rows
        h += "<tr>" + "".join(f"<td style='{TH}text-align:{'left;white-space:pre' if j == 0 else 'center'};{'font-weight:bold;background:#f2f2f2' if sec else 'background:#fff'}'>{v}</td>" for j, v in enumerate(row)) + "</tr>"
    return HTML(h + "</table>")
num = lambda r, key: float(re.search(rf"{key}=([\d.]+)", r["result"]).group(1)) if re.search(rf"{key}=([\d.]+)", r["result"]) else np.nan
ok = lambda k: fits[k]["status"] == "ok" and os.path.exists(out[k])
fixed_lab = {"age_z": "Age at visit (per SD)", "year_z": "Calendar year (per SD)", "teeth_z": "Teeth at visit (per SD)", "logdays_z": "log(time since preceding visit) (per SD)",
             "cls2": "Severity class 2 (vs. 1)", "cls3": "Severity class 3 (vs. 1)", "female": "Female (vs. male)", "ever": "Ever-smoker (vs. never)",
             "cls2_female": "Class 2 x female", "cls3_female": "Class 3 x female"}
var_lab = {"sigma": "Residual SD", **{f"sd_intercept, {g}": f"Random intercept SD, {g}" for g in sorted(s["grp_re"].unique())}, "sd_age": "Random slope SD (age)", "sd_teeth": "Random slope SD (teeth)", "sd_logdays": "Random slope SD (log time since preceding visit)"}
def compare(tk, ok_key, olabel, title):
    keys = {"t": tk, "o": ok_key}
    print(f"\n=== {title} ===")
    fmt = lambda v: "n/a" if pd.isna(v) else f"{v:.3g}" if abs(v) < 1 else f"{v:.2f}"
    rat = lambda a, b: "n/a" if pd.isna(a) or pd.isna(b) else f"{a / b:.2g}"
    met = {"Total wall time (s)": [fits[k]["wall (s)"] if fits[k]["status"] == "ok" else np.nan for k in keys.values()],
           "Model fitting (s)": [num(fits[k], "fit_seconds") for k in keys.values()],
           "Summary and tests (s)": [num(fits[k], "summary_seconds") for k in keys.values()],
           "Peak memory (GB)": [fits[k]["peak memory (GB)"] for k in keys.values()],
           "Smooth coefficients": [num(fits[tk], "smooth_coefs"), num(fits[ok_key], "smooth_basis")]}
    t1 = pd.DataFrame([[m, fmt(v[0]), fmt(v[1]), rat(v[1], v[0])] for m, v in met.items()], columns=["Metric", "torchpalmm", olabel, f"{olabel.split('::')[-1]} / torchpalmm"])
    cv = lambda k: "timeout" if fits[k]["status"] == "timeout" else "error" if fits[k]["status"] != "ok" else "Yes" if num(fits[k], "converged") == 1 else "No"
    t1.loc[len(t1)] = ["Converged", cv(tk), cv(ok_key), ""]
    display(html_table(t1))
    if not (ok(tk) and ok(ok_key)):
        print("one of the two fits did not finish: no parameter table"); return
    e = {k: pd.read_csv(out[k]).set_index("term") for k in keys.values()}
    d = {k: pd.read_csv(out[k].replace(".csv", "_edf.csv")).set_index("term")["edf"] for k in keys.values()}
    cell = lambda k, t, se=True: "n/a" if pd.isna(e[k].loc[t, "estimate"]) else f"{e[k].loc[t, 'estimate']:.3f} ({e[k].loc[t, 'se']:.3f})" if se and not pd.isna(e[k].loc[t, "se"]) else f"{e[k].loc[t, 'estimate']:.3f}"
    dif = lambda a, b, f_: "n/a" if pd.isna(a) or pd.isna(b) else f_.format(a - b)
    rows, sec = [["Fixed effects, estimate (SE)", "", "", ""]], [0]
    for t, lab in fixed_lab.items():
        rows.append(["    " + lab, cell(tk, t), cell(ok_key, t), dif(e[tk].loc[t, "estimate"], e[ok_key].loc[t, "estimate"], "{:.1e}")])
    rows.append(["Smooth terms, EDF", "", "", ""]); sec.append(len(rows) - 1)
    for t in d[tk].index:
        
        a, b = d[tk].get(t, np.nan), d[ok_key].get(t, np.nan)
        rows.append(["    " + t, fmt(a), fmt(b), dif(a, b, "{:.2f}")])
    rows.append(["Variance components", "", "", ""]); sec.append(len(rows) - 1)
    for t, lab in var_lab.items():
        rows.append(["    " + lab, cell(tk, t, False), cell(ok_key, t, False), dif(e[tk].loc[t, "estimate"], e[ok_key].loc[t, "estimate"], "{:.1e}")])
    display(html_table(pd.DataFrame(rows, columns=["Parameter", "torchpalmm", olabel, f"Diff. vs {olabel.split('::')[-1]}"]), set(sec)))
compare("torchpalmm", "gamm", "mgcv::gamm", "torchpalmm vs mgcv::gamm")
compare("torchpalmm", "bam", "mgcv::bam", "torchpalmm vs mgcv::bam")



=== torchpalmm vs mgcv::gamm ===


Metric,torchpalmm,mgcv::gamm,gamm / torchpalmm
Total wall time (s),504.71,n/a,n/a
Model fitting (s),172.20,n/a,n/a
Summary and tests (s),331.40,n/a,n/a
Peak memory (GB),2.99,n/a,n/a
Smooth coefficients,1515.00,n/a,n/a
Converged,Yes,timeout,


one of the two fits did not finish: no parameter table

=== torchpalmm vs mgcv::bam ===


Metric,torchpalmm,mgcv::bam,bam / torchpalmm
Total wall time (s),504.71,n/a,n/a
Model fitting (s),172.20,n/a,n/a
Summary and tests (s),331.40,n/a,n/a
Peak memory (GB),2.99,n/a,n/a
Smooth coefficients,1515.00,n/a,n/a
Converged,Yes,timeout,


one of the two fits did not finish: no parameter table
